In [5]:
from bstabdiff import BlockSubunitGenerator, FeatureSpec, fit_block_subunit_generator

# Dummy Example 1

In [3]:
import numpy as np
from bstabdiff import FeatureSpec, fit_block_subunit_generator

# -------------------------
# 1) Create a dummy HDLSS dataset
# -------------------------
np.random.seed(42)

n = 80    # number of samples
m = 200   # number of features (HDLSS: n << m)

X = np.random.randn(n, m).astype(np.float32)

# binary labels
y = np.random.randint(0, 2, size=n)

# add some missingness
missing_mask = np.random.rand(n, m) < 0.1
X[missing_mask] = np.nan

# -------------------------
# 2) Define feature schema
# -------------------------
feature_specs = [
    FeatureSpec(name=f"f{j}", kind="continuous")
    for j in range(m)
]

# -------------------------
# 3) Fit BSTabDiff
# -------------------------
gen, train_info = fit_block_subunit_generator(
    X=X,
    feature_specs=feature_specs,
    y=y,
    M=20,                       # latent block dimension
    prior_type="diffusion",     # or "flow"
    device="cpu",               # change to "cuda:0" if available
    seed=42,
    prior_epochs=300,
    prior_batch=64,
    prior_lr=1e-3,
    verbose_every=100,
    save_dir=None,
    save_name="bstabdiff_demo",
    save_best=True,
    use_ema=True,
    ema_decay=0.999,
    return_train_info=True,
)

print("Training info:")
print(train_info)

# -------------------------
# 4) Sample synthetic data
# -------------------------
X_syn, R_syn, y_syn = gen.sample(n=50)

print("\nSynthetic outputs:")
print("X_syn shape:", X_syn.shape)
print("R_syn shape:", R_syn.shape)
print("y_syn shape:", None if y_syn is None else y_syn.shape)

print("\nFirst 5 synthetic labels:")
print(y_syn[:5] if y_syn is not None else None)

print("\nObserved fraction in synthetic mask:")
print(R_syn.mean())

print("\nFirst synthetic sample (first 10 features):")
print(X_syn[0, :10])

[prior:diffusion] epoch 1/300 | loss=1.090266
[prior:diffusion] epoch 100/300 | loss=0.581224
[prior:diffusion] epoch 200/300 | loss=0.566903
[prior:diffusion] epoch 300/300 | loss=0.575131
Training info:
{'best_epoch': 245, 'best_loss': 0.4641336500644684, 'best_ckpt_path': None, 'loaded_at_end': True, 'prior_type': 'diffusion', 'used_ema': True, 'ema_decay': 0.999}

Synthetic outputs:
X_syn shape: (50, 200)
R_syn shape: (50, 200)
y_syn shape: (50,)

First 5 synthetic labels:
[0 0 1 1 1]

Observed fraction in synthetic mask:
0.8958

First synthetic sample (first 10 features):
[ 0.04739276 -0.5970627  -0.41054967 -1.373018   -0.9582272  -1.8693262
         nan -0.9617864  -0.42801404 -0.1865349 ]


# Dummy Example 2

In [4]:
import numpy as np
from bstabdiff import (
    FeatureSpec,
    fit_block_subunit_generator,
)

# =========================================================
# 1) Create a dummy HDLSS dataset
# =========================================================
np.random.seed(42)

n = 80                    # number of samples
m = 200                   # number of features (HDLSS: n << m)
n_classes = 2             # binary classification example
missing_rate = 0.10       # fraction of missing entries

# Real data matrix
X = np.random.randn(n, m).astype(np.float32)

# Labels
y = np.random.randint(0, n_classes, size=n)

# Add missing values
missing_mask = np.random.rand(n, m) < missing_rate
X[missing_mask] = np.nan

# =========================================================
# 2) Define feature schema
# =========================================================
# Here all features are continuous.
# For categorical features, use:
# FeatureSpec(name="f0", kind="categorical", n_categories=3)
feature_specs = [
    FeatureSpec(name=f"f{j}", kind="continuous")
    for j in range(m)
]

# =========================================================
# 3) BSTabDiff fitting configuration
# =========================================================
M = 20                         # number of latent blocks
blocks = None                  # None -> uses equal block partition automatically
permute_features = False       # whether to apply a random feature permutation
prior_type = "diffusion"       # choices: "diffusion" or "flow"
device = "cpu"                 # use "cuda:0" if GPU is available
seed = 42

# Prior training hyperparameters
prior_epochs = 300
prior_batch = 64
prior_lr = 1e-3
verbose_every = 100

# Checkpoint / best-model options
save_dir = None                # e.g. "checkpoints" to save best checkpoint
save_name = "bstabdiff_demo"
save_best = True

# EMA options (mainly useful for diffusion prior)
use_ema = True
ema_decay = 0.999

# Whether to return training diagnostics
return_train_info = True

# =========================================================
# 4) Fit BSTabDiff
# =========================================================
gen, train_info = fit_block_subunit_generator(
    X=X,
    feature_specs=feature_specs,
    y=y,
    M=M,
    blocks=blocks,
    permute_features=permute_features,
    prior_type=prior_type,
    device=device,
    seed=seed,
    prior_epochs=prior_epochs,
    prior_batch=prior_batch,
    prior_lr=prior_lr,
    verbose_every=verbose_every,
    save_dir=save_dir,
    save_name=save_name,
    save_best=save_best,
    use_ema=use_ema,
    ema_decay=ema_decay,
    return_train_info=return_train_info,
)

print("Training info:")
print(train_info)

# =========================================================
# 5) Sample synthetic data
# =========================================================
n_syn = 50

# If y is omitted, class labels are sampled automatically (when class-conditional)
X_syn, R_syn, y_syn = gen.sample(n=n_syn)

print("\nSynthetic outputs:")
print("X_syn shape:", X_syn.shape)   # synthetic data
print("R_syn shape:", R_syn.shape)   # observed/missing mask (1=observed, 0=missing)
print("y_syn shape:", None if y_syn is None else y_syn.shape)

print("\nFirst 5 synthetic labels:")
print(y_syn[:5] if y_syn is not None else None)

print("\nObserved fraction in synthetic mask:")
print(R_syn.mean())

print("\nFirst synthetic sample (first 10 features):")
print(X_syn[0, :10])

# =========================================================
# 6) Optional: sample from a fixed class
# =========================================================
X_syn_c1, R_syn_c1, y_syn_c1 = gen.sample(n=10, y=1)

print("\nFixed-class synthetic labels:")
print(y_syn_c1)

# =========================================================
# 7) Optional: inspect learned model structure
# =========================================================
print("\nModel summary:")
print("Number of features (m):", gen.m)
print("Number of blocks (M):", gen.M)
print("Prior type:", gen.prior_type)
print("Has permutation:", gen.perm is not None)

[prior:diffusion] epoch 1/300 | loss=1.090266
[prior:diffusion] epoch 100/300 | loss=0.581224
[prior:diffusion] epoch 200/300 | loss=0.566903
[prior:diffusion] epoch 300/300 | loss=0.575131
Training info:
{'best_epoch': 245, 'best_loss': 0.4641336500644684, 'best_ckpt_path': None, 'loaded_at_end': True, 'prior_type': 'diffusion', 'used_ema': True, 'ema_decay': 0.999}

Synthetic outputs:
X_syn shape: (50, 200)
R_syn shape: (50, 200)
y_syn shape: (50,)

First 5 synthetic labels:
[0 0 1 1 1]

Observed fraction in synthetic mask:
0.8958

First synthetic sample (first 10 features):
[ 0.04739276 -0.5970627  -0.41054967 -1.373018   -0.9582272  -1.8693262
         nan -0.9617864  -0.42801404 -0.1865349 ]

Fixed-class synthetic labels:
[1 1 1 1 1 1 1 1 1 1]

Model summary:
Number of features (m): 200
Number of blocks (M): 20
Prior type: diffusion
Has permutation: False
